# Credit Card Fraud Detection — Exploratory Data Analysis

This notebook explores the transaction dataset used by the project. It loads data
through the same `src` modules the pipeline uses, so it works with either the real
Kaggle dataset or the synthetic fallback (no credentials required).

**Goals:** quantify the class imbalance, inspect the `Amount` / `Time` distributions,
and look at which features separate fraud from legitimate transactions.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/.
sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import seaborn as sns

from src.config import load_config
from src.data.loader import load_data

sns.set_theme(style="whitegrid")
config = load_config()
df = load_data(config)
df.shape

In [ ]:
df.head()

## 1. Class imbalance

Fraud is extremely rare. This is why we optimise for PR-AUC / recall rather than
accuracy, and why we apply SMOTE to the training split only.

In [ ]:
target = config.data.target_column
counts = df[target].value_counts()
print(counts)
print(f"Fraud rate: {df[target].mean() * 100:.3f}%")

ax = counts.plot(kind="bar", color=["#4c72b0", "#c44e52"])
ax.set_xticklabels(["Legitimate", "Fraud"], rotation=0)
ax.set_title("Class distribution")
ax.set_ylabel("Count")
plt.show()

## 2. Transaction amount distribution

Fraudulent transactions often have a different amount profile from legitimate ones.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for cls, ax in zip([0, 1], axes):
    subset = df[df[target] == cls]["Amount"]
    ax.hist(subset, bins=50, color="#4c72b0" if cls == 0 else "#c44e52")
    ax.set_title(f"Amount — {'Legitimate' if cls == 0 else 'Fraud'}")
    ax.set_xlabel("Amount")
    ax.set_yscale("log")
plt.tight_layout()
plt.show()

df.groupby(target)["Amount"].describe()

## 3. Feature correlation with the target

Which features carry the most signal about fraud?

In [ ]:
correlations = df.corr(numeric_only=True)[target].drop(target).sort_values(key=abs, ascending=False)
top = correlations.head(10)
print(top)

ax = top.plot(kind="barh", color="#4c72b0")
ax.invert_yaxis()
ax.set_title("Top 10 features by |correlation| with fraud")
ax.set_xlabel("Correlation with target")
plt.tight_layout()
plt.show()

## Takeaways

- The dataset is severely imbalanced (~0.17% fraud), so accuracy is not a useful metric.
- A handful of PCA components carry most of the fraud signal; the rest are near-noise.
- `Amount` and `Time` are on a different scale from the PCA components, so they are
  standardised inside the modelling pipeline.

Next step: run `python main.py` to train and compare all five models.